# Modeling — Baseline CNN, Transfer Learning, and Hyperparameter Search
### Early Dysgraphia Screening from Handwriting — Potential Dysgraphia Handwriting Dataset (PDM)

**CRISP-DM phase:** Modeling

**Purpose of this notebook**

Steps 1–3 gave us a clean, standardized image pipeline (`preprocess_image`) and two interpretable indicators (spacing consistency, baseline deviation). This notebook covers the Modeling phase from the project write-up:

1. **Baseline CNN** trained from scratch — a diagnostic floor the transfer learning model needs to beat, and a check that the whole pipeline works end to end.
2. **Transfer learning model** — a pretrained backbone with a frozen-then-unfrozen training schedule, dropout in the head, and a learning rate schedule.
3. **Hyperparameter search** — a small manual grid search over learning rate, batch size, dropout rate, and unfreezing depth, guided by **validation recall on the Potential Dysgraphia class**.
4. **Threshold Calibration** — abandoning raw probability outputs in favor of Youden's J statistic to balance recall and precision, ensuring the model doesn't just panic and classify everything as a positive case.

> **Before running:** point `DATASET_ROOT` at the folder that directly contains the two class subfolders (`Low Potential Dysgraphia/` and `Potential Dysgraphia/`).

## 2. Load the Dataset

The dataset consists of 1,625 images[cite: 4]. We run every image through `preprocess_image` once here, then split.

We use a **60 / 20 / 20 stratified split** into train (975), validation (325), and test (325) sets[cite: 4]. Stratifying keeps the roughly 64.5% class balance consistent across all three sets[cite: 4]. The test set is set aside now and only touched once, at the very end, for the final honest check — matching the plan in the project write-up.

In [ ]:
# imports
import os, hashlib, random
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score, f1_score, recall_score, precision_score,
)

# Set the random seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

CLASSES = ["Low Potential Dysgraphia", "Potential Dysgraphia"]

# Potential Dysgraphia encoded it as the positive class (1).
LABEL_MAP = {"Low Potential Dysgraphia": 0, "Potential Dysgraphia": 1}
POSITIVE_CLASS = "Potential Dysgraphia"

# Dataset root folder
DATASET_ROOT = Path("DATASET DYSGRAPHIA HANDWRITING")
assert DATASET_ROOT.exists(), (
    f"Could not find '{DATASET_ROOT}'. Update DATASET_ROOT to point at the "
    f"unzipped dataset folder (the one that directly contains the class subfolders)."
)

# width, height fed into preprocess_image
TARGET_SIZE = (256, 256)

2026-09-08 15:46:37.156669: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-08 15:46:38.036914: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-09-08 15:46:41.943696: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## 2. Load the Dataset and Build Stratified Splits

At 249 images total, the whole dataset comfortably fits in memory as a single numpy
array, so there's no need for a lazy-loading `tf.data` pipeline just to manage size. We
run every image through `preprocess_image` once here, then split.

We use a **60 / 20 / 20 stratified split** into train / validation / test. Stratifying
keeps the roughly 135:114 class balance consistent across all three sets, and the test
set is set aside now and only touched once, at the very end, for the final honest check —
matching the plan in the project write-up.


In [ ]:
# build a file index
records = []
for cls in CLASSES:
    folder = DATASET_ROOT / cls
    for fname in sorted(os.listdir(folder)):
        if fname.lower().endswith((".jpg", ".jpeg", ".png")):
            records.append({"path": str(folder / fname), "class": cls, "label": LABEL_MAP[cls]})

index_df = pd.DataFrame(records)
print("Total images:", len(index_df))
print(index_df["class"].value_counts())

Total images: 1625
class
Potential Dysgraphia        1049
Low Potential Dysgraphia     576
Name: count, dtype: int64
